# 一、 概念

核心问题：为什么必须有激活函数？<br>
"线性层叠线性层 = 还是线性，所以必须有非线性激活函数。"<br>
无激活函数：多层全连接等价单层线性变换，网络深度作废，只能拟合线性关系。激活函数引入非线性（如 ReLU）,让深层网络拥有拟合复杂数据的能力。

把它想透，这节就差不多了，推理链条：<br>
一个线性层做的事情是 Wx + b，本质是矩阵乘法 + 平移。<br>
如果叠两层、三层、N 层，中间不做任何非线性变换，那么：<br>
第一层：h = W₁x + b₁<br>
第二层：o = W₂h + b₂<br>
如果两层之间没有非线性：<br>
o = W₂(W₁x + b₁) + b₂ = (W₂W₁)x + (W₂b₁ + b₂)
                        ↑新的一个矩阵      ↑一个新的向量<br>  
W₂W₁ 仍然是一个矩阵，W₂b₁ + b₂ 仍然是一个向量。<br>
结论：任意深的线性层叠 = 一个线性层。表达能力没有任何增加。<br>
所以中间必须插入非线性激活函数，把每一层"打断"，让网络真正变"深"。

两个推论：<br>
为什么激活函数必须是逐元素（element-wise）的？因为逐元素非线性无法被吸收进矩阵乘法里。
感知机（单层）失败于异或（XOR）——(0,1)、(1,0) 是一类，(0,0)、(1,1) 是另一类，没有任何一条直线能分开它们。根源就是没有隐藏层 + 没有非线性——这是历史动机，所以这就是为什么 MLP 是"多层感知机"。"多层"感知机（MLP）存在的理由：用隐藏层把输入空间扭曲到线性可分。

三个激活函数对比（只需记住特点）：<br>
函数公式特点何时用sigmoid1/(1+e⁻ˣ)饱和区梯度消失；输出非零中心输出层二分类概率<br>tanh(eˣ−e⁻ˣ)/(eˣ+e⁻ˣ)零中心，但仍饱和早年间隐藏层ReLUmax(0, x)不饱和（x>0 梯度恒 1）；计算快默认隐藏层首选<br>
ReLU 唯一的坑：x<0 时梯度为 0，可能"神经元死亡"

网络结构：<br>
输入 784 (28×28 展平)<br>
  → W1(784×256), b1(256)  → 隐藏层 256 个神经元 → ReLU<br>
  → W2(256×10),  b2(10)   → 输出层 10 个 logits（不加激活，softmax 交给损失函数）

# 二、从零实现

## 1：数据 + 参数初始化

In [16]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
num_inputs, num_outputs, num_hiddens = 784, 10, 256
W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens))
W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs))
params = [W1, b1, W2, b2]
'''
W1 是 784×256，W2 是 256×10——这就是"线性层叠"的维度传递，和概念部分呼应。
'''

'\nW1 是 784×256，W2 是 256×10——这就是"线性层叠"的维度传递，和概念部分呼应。\n'

## 2：ReLU + 前向

In [17]:
def relu(X):
    return torch.max(X, torch.zeros_like(X))

def net(X):
    X = X.reshape((-1, num_inputs))
    H = relu(X @ W1 + b1)  # ReLU 激活：引入非线性，过滤负值，得到隐藏特征 H
    return H @ W2 + b2   # 输出层不加激活，logits 直接进损失
'''
训练闭环的复用：这里不写 softmax，用带 reduction='none' 的交叉熵损失（它内部会算 softmax）——省一步，也避免数值不稳定。这是"简洁版"和"从零版"的第一次融合。
'''

'\n训练闭环的复用：softmax 已经在 ch3 做过，这里不写 softmax，用带 reduction=\'none\' 的交叉熵损失（它内部会算 softmax）——省一步，也避免数值不稳定。这是"简洁版"和"从零版"的第一次融合。\n'

## 3：损失 + 训练

In [18]:
# 4. 损失：reduction='none'（逐样本），backward 时用 mean —— d2l 标准
loss = nn.CrossEntropyLoss(reduction='none')
# 5. 训练循环（注意：l.mean().backward()，不是 sum！）
def train_epoch(net, train_iter, loss, updater):
    total_loss, total_acc, n = 0.0, 0.0, 0
    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y)
        updater.zero_grad()
        l.mean().backward()      # ← 关键：mean，不是 sum
        updater.step()
        total_loss += l.sum().item()   # 统计时用 sum 没问题
        total_acc += (y_hat.argmax(1) == y).sum().item()
        n += y.shape[0]
    return total_loss / n, total_acc / n
def evaluate_accuracy(net, data_iter):
    acc, n = 0.0, 0
    for X, y in data_iter:
        acc += (net(X).argmax(1) == y).sum().item()
        n += y.shape[0]
    return acc / n
def train(net, train_iter, test_iter, loss, num_epochs, updater):
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        print(f'epoch {epoch + 1}: loss={train_loss:.4f}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}')
num_epochs, lr = 10, 0.1
updater = torch.optim.SGD(params, lr=lr)
train(net, train_iter, test_iter, loss, num_epochs, updater)

epoch 1: loss=1.0404, train_acc=0.6449, test_acc=0.7276
epoch 2: loss=0.6031, train_acc=0.7888, test_acc=0.7940
epoch 3: loss=0.5199, train_acc=0.8192, test_acc=0.8234
epoch 4: loss=0.4807, train_acc=0.8319, test_acc=0.8092
epoch 5: loss=0.4525, train_acc=0.8414, test_acc=0.8272
epoch 6: loss=0.4344, train_acc=0.8468, test_acc=0.8348
epoch 7: loss=0.4192, train_acc=0.8535, test_acc=0.8381
epoch 8: loss=0.4022, train_acc=0.8580, test_acc=0.8462
epoch 9: loss=0.3930, train_acc=0.8617, test_acc=0.8439
epoch 10: loss=0.3812, train_acc=0.8653, test_acc=0.8150


# 三、小结

1. 为什么 W 用 randn * 0.01 而 b 用 zeros？如果所有 W 初始化为 0 会怎样？（提示：对称性问题）<br>
W 为权重，randn 生成的是标准正态数据，randn*0.01 让初始权重在小范围随机波动（打破对称），而b 用 zeros 因为偏置全 0 不影响对称性，b 只是管平移而已；如果 W 全部初始化为 0 的话那么隐藏层输出都为0；<br>
对称性——全零初始化时，所有隐藏单元的输出相同、梯度相同，永远同步更新、永远学不到不同的特征，网络退化成"一个神经元"。
  
2. relu 为什么不能写成 X.clamp(min=0) 之外的其他写法？两者等价吗？<br>
你的 relu 不是用的 max 函数吗，也是取得比 0 大的值，如果 X 为 0 的话取0 ，这个.clamp(min=0)的作用：把张量里所有小于 0 的元素全部截断变成 0；大于等于 0 的保持不变。两个的效果是一样的吧

3. reduction='none' 是干嘛的？训练时到底用 sum 还是 mean？<br>
不做聚合，返回每个样本各自的损失,长度 = batch_size，每个元素是单个样本的损失<br>
训练时用 mean，原因一句话：br>

| reduction | 返回 | 含义 |
|-----------|------|------|
| `'none'` | 长度=batch_size 的向量 | **每个样本单独一个损失**，不聚合、不平均 |
| `'mean'`（默认） | 标量 | 所有样本损失求和 ÷ batch_size |
| `'sum'` | 标量 | 所有样本损失直接求和 |

optimizer（SGD）内部不做任何归一化，只做 w -= lr × 梯度。梯度是从损失算出来的——用 mean，梯度是平均量级，lr=0.1 正好；用 sum，梯度放大 batch 倍，lr 必须跟着缩小 batch 倍，否则就爆炸

4. 什么是对称？为什么要打破对称？<br>
同一层所有神经元的权重全部一模一样，这就叫对称。<br>
举个例子：隐藏层 2 个神经元<br>
神经元 1 权重：[0.01, 0.02]<br>
神经元 2 权重：完全和 1 一模一样 [0.01,0.02]<br>
两个神经元天生就是双胞胎，参数完全相同。

 如果不打破对称（W 全部初始为 0 就会出现）<br>
 前向传播：输入给两个神经元，算出来输出完全相等。<br>
 反向传播：算出来的梯度也完全相等。<br>
 参数更新：w = w - lr*grad<br>
 两个神经元更新的数值一模一样。<br>
 训练一万轮，两个神经元永远一模一样